# 16 — Final Model Training & Pipeline Serialization

**Objective:** Freeze and serialize the production-ready model pipeline (`final_model.joblib`) and its complete operational metadata.


In [1]:
import sys
import json
import time
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.models.model_factory import create_model
from src.data.loader import FEATURES_A, TARGETS_A
from src.data.splitting import temporal_split
from src.evaluation.conformal import conformal_quantiles


## 1. Freezing Final Candidate Pipeline


In [2]:
df = pd.read_parquet(ROOT / "data" / "processed" / "cleaned_master.parquet")
dev, cal, test = temporal_split(df)

# Train production model (HistGradientBoosting) on development data
final_pipeline = create_model("HistGradientBoosting", quick=False, random_state=42)
final_pipeline.fit(dev[FEATURES_A], dev[TARGETS_A].values)

# Calibrate conformal quantiles
pred_cal = final_pipeline.predict(cal[FEATURES_A])
q = conformal_quantiles(cal[TARGETS_A].values, pred_cal, alpha=0.10)

bundle = {
    "model_name": "HistGradientBoosting_Production",
    "estimator": final_pipeline,
    "features": FEATURES_A,
    "targets": TARGETS_A,
    "conformal_alpha": 0.10,
    "conformal_quantiles": q,
    "training_date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "training_rows": len(dev),
    "calibration_rows": len(cal)
}

out_model = ROOT / "models" / "final_model.joblib"
out_model.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, out_model)

metadata = {
    "model_name": bundle["model_name"],
    "features": bundle["features"],
    "targets": bundle["targets"],
    "conformal_alpha": bundle["conformal_alpha"],
    "conformal_quantiles": bundle["conformal_quantiles"],
    "training_date": bundle["training_date"],
    "training_rows": bundle["training_rows"]
}

out_meta = ROOT / "models" / "model_metadata.json"
with open(out_meta, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved serialized final model to {out_model.relative_to(ROOT)} ({out_model.stat().st_size:,} bytes)")
print(f"Saved model metadata to {out_meta.relative_to(ROOT)}")
print("Final model training and serialization COMPLETE.")


Saved serialized final model to models\final_model.joblib (1,785,833 bytes)
Saved model metadata to models\model_metadata.json
Final model training and serialization COMPLETE.
